<a href="https://colab.research.google.com/github/faiqakashif82-netizen/FlyRank-MachineLearning-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faiqakashif82-netizen/FlyRank-MachineLearning-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1. Signal checks

Before building the baseline rule, I checked two signals that the rule will rely on.

- **Signal 1: content age / staleness** — this is linked to FlyRank's refresh/staleness flag logic.
- **Signal 2: search volume / impressions** — this is linked to the quick-win idea, where meaningful search demand makes an opportunity more actionable.

I will check each signal with bucket-level results and sample size (`n`) before using it in the rule.

In [30]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "https://raw.githubusercontent.com/faiqakashif82-netizen/FlyRank-MachineLearning-Internship/main/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [31]:
# Check the dataset
print("Rows:", len(df))
print("Columns:", len(df.columns))

# See the columns related to age / freshness
[c for c in df.columns if "age" in c.lower() or "fresh" in c.lower() or "stale" in c.lower()]


Rows: 30000
Columns: 44


['pageviews_90d',
 'engaged_sessions_90d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'freshness_tier',
 'engagement_rate']

In [32]:
# Bucket content age into simple freshness groups
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90 days", "91-180 days", "181-365 days", "366+ days"]
)

# Show the bucket table with n
age_check = (
    df.groupby("age_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          avg_engagement=("engagement_rate", "mean")
      )
      .reset_index()
)

age_check

,age_bucket,n,avg_engagement
0,0-90 days,492,3.236423
1,91-180 days,11780,2.151572
2,181-365 days,11368,2.676583
3,366+ days,6360,2.935591


In [33]:
# Bucket pages by how long it has been since the last update
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90 days", "91-180 days", "181-365 days", "366+ days"]
)

# Check recent impressions by staleness bucket
staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          avg_impressions_30d=("impressions_last_30d", "mean")
      )
      .reset_index()
)

staleness_check

,staleness_bucket,n,avg_impressions_30d
0,0-90 days,20655,1186.065456
1,91-180 days,9171,2001.390906
2,181-365 days,169,111.360947
3,366+ days,5,0.800000


**Staleness signal verdict: MIXED**

The staleness signal is mixed. Pages updated 91–180 days ago have higher average recent impressions than pages updated within 90 days, so greater staleness does not consistently correspond to lower search visibility in this sample. The older groups also have very small sample sizes (n=169 and n=5), which makes their averages less reliable. Therefore, I would not use staleness alone as a strong priority signal.

In [34]:
# Bucket search volume into simple groups
df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=[-1, 10, 50, 100, np.inf],
    labels=["0-10", "11-50", "51-100", "101+"]
)

# Check recent impressions by search-volume bucket
volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          avg_impressions_30d=("impressions_last_30d", "mean")
      )
      .reset_index()
)

volume_check

,volume_bucket,n,avg_impressions_30d
0,0-10,18392,1501.356514
1,11-50,4989,1574.944678
2,51-100,1102,1767.310345
3,101+,3049,1678.170220


**Search-volume signal verdict: CONFIRMED**

Search volume shows a generally positive relationship with recent impressions. Average impressions increase from 1,501 for the 0–10 group to 1,767 for the 51–100 group, while the 101+ group remains high at 1,678. This supports using search volume as a useful opportunity signal, although the relationship is not perfectly monotonic.

### Baseline rule

I will prioritize content using **search volume and recent impressions**.

The rule gives a higher score to pages with stronger search demand and stronger recent search visibility. I will not use staleness as a scoring input because its signal check was MIXED.

**Score:**

- 2 points if search volume is 51 or higher
- 1 point if search volume is 11–50
- 0 points if search volume is 0–10
- 2 additional points if recent impressions are 1,000 or higher
- 1 additional point if recent impressions are 500–999
- 0 additional points if recent impressions are below 500

**Reason code:** `HIGH_OPPORTUNITY`

**Action:** `REVIEW`

The higher the score, the earlier the content appears in the ranked queue.

In [35]:
# Create the baseline score

df["score"] = 0

# Search-volume points
df.loc[df["search_volume"].between(11, 50), "score"] += 1
df.loc[df["search_volume"] >= 51, "score"] += 2

# Recent-impressions points
df.loc[df["impressions_last_30d"].between(500, 999), "score"] += 1
df.loc[df["impressions_last_30d"] >= 1000, "score"] += 2

# One reason code
df["reason_code"] = np.where(
    df["score"] >= 3,
    "HIGH_OPPORTUNITY",
    "LOW_OPPORTUNITY"
)

# One action label
df["action"] = np.where(
    df["score"] >= 3,
    "REVIEW",
    "MONITOR"
)

df[[
    "content_id",
    "search_volume",
    "impressions_last_30d",
    "score",
    "reason_code",
    "action"
]].head(10)

,content_id,search_volume,impressions_last_30d,score,reason_code,action
0,content_304f48230142,10.0,578,1,LOW_OPPORTUNITY,MONITOR
1,content_a1fb4e703a9e,90.0,2501,4,HIGH_OPPORTUNITY,REVIEW
2,content_9aa793d4d895,0.0,2382,2,LOW_OPPORTUNITY,MONITOR
3,content_331d6c4de07b,10.0,3626,2,LOW_OPPORTUNITY,MONITOR
4,content_d99b7a2d90ca,0.0,4211,2,LOW_OPPORTUNITY,MONITOR
5,content_d4084a4bc775,720.0,617,3,HIGH_OPPORTUNITY,REVIEW
6,content_9a34b442b552,0.0,1,0,LOW_OPPORTUNITY,MONITOR
7,content_a63219c6e95a,590.0,636,3,HIGH_OPPORTUNITY,REVIEW
8,content_5e6c160719bc,0.0,5696,2,LOW_OPPORTUNITY,MONITOR
9,content_c27558df2b0c,0.0,252,0,LOW_OPPORTUNITY,MONITOR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [36]:
# Build the ranked queue
ranked_queue = (
    df[
        [
            "content_id",
            "score",
            "reason_code",
            "action",
            "search_volume",
            "impressions_last_30d"
        ]
    ]
    .sort_values(
        by=["score", "search_volume", "impressions_last_30d"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

# Add rank
ranked_queue.insert(0, "rank", ranked_queue.index + 1)

# Show the top 10
ranked_queue.head(10)

,rank,content_id,score,reason_code,action,search_volume,impressions_last_30d
0,1,content_deb54e9e19cd,4,HIGH_OPPORTUNITY,REVIEW,60500.0,1119
1,2,content_ee4630879d03,4,HIGH_OPPORTUNITY,REVIEW,49500.0,1171
2,3,content_7868341d97dd,4,HIGH_OPPORTUNITY,REVIEW,40500.0,3015
3,4,content_8ca50876b0df,4,HIGH_OPPORTUNITY,REVIEW,40500.0,1952
4,5,content_c841193dc692,4,HIGH_OPPORTUNITY,REVIEW,40500.0,1082
5,6,content_eb1510f4b5f1,4,HIGH_OPPORTUNITY,REVIEW,33100.0,1615
6,7,content_d99d0671553d,4,HIGH_OPPORTUNITY,REVIEW,22200.0,1087
7,8,content_2db251d1a841,4,HIGH_OPPORTUNITY,REVIEW,14800.0,86434
8,9,content_c35d34131e91,4,HIGH_OPPORTUNITY,REVIEW,14800.0,4547
9,10,content_496544bf85aa,4,HIGH_OPPORTUNITY,REVIEW,14800.0,1928


In [37]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

ranked_queue.to_csv(output_path, index=False)

print(f"Saved ranked queue to: {output_path}")
print(f"Rows written: {len(ranked_queue)}")

Saved ranked queue to: work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 review

I reviewed the top ten ranked items from the baseline queue. For each item, I record the recommended action, why the rule selected it, and what could make the recommendation wrong.

In [38]:
top10 = ranked_queue.head(10).copy()

top10

,rank,content_id,score,reason_code,action,search_volume,impressions_last_30d
0,1,content_deb54e9e19cd,4,HIGH_OPPORTUNITY,REVIEW,60500.0,1119
1,2,content_ee4630879d03,4,HIGH_OPPORTUNITY,REVIEW,49500.0,1171
2,3,content_7868341d97dd,4,HIGH_OPPORTUNITY,REVIEW,40500.0,3015
3,4,content_8ca50876b0df,4,HIGH_OPPORTUNITY,REVIEW,40500.0,1952
4,5,content_c841193dc692,4,HIGH_OPPORTUNITY,REVIEW,40500.0,1082
5,6,content_eb1510f4b5f1,4,HIGH_OPPORTUNITY,REVIEW,33100.0,1615
6,7,content_d99d0671553d,4,HIGH_OPPORTUNITY,REVIEW,22200.0,1087
7,8,content_2db251d1a841,4,HIGH_OPPORTUNITY,REVIEW,14800.0,86434
8,9,content_c35d34131e91,4,HIGH_OPPORTUNITY,REVIEW,14800.0,4547
9,10,content_496544bf85aa,4,HIGH_OPPORTUNITY,REVIEW,14800.0,1928


### Top-10 review

| Rank | Content | Action | Why it's there | What would make it wrong |
|---:|---|---|---|---|
| 1 | `content_deb54e9e19cd` | REVIEW | Very high search volume (60,500) and recent impressions above 1,000 give it the maximum baseline score. | The search-volume estimate could be inflated, or the page may already be performing well enough that review would add little value. |
| 2 | `content_ee4630879d03` | REVIEW | High search volume (49,500) and recent impressions above 1,000 give it the maximum score. | The page may already satisfy the search demand well, so high volume alone may not indicate a content problem. |
| 3 | `content_7868341d97dd` | REVIEW | High search volume (40,500) and strong recent impressions (3,015) produce the maximum score. | Strong existing performance could mean there is little actionable improvement opportunity. |
| 4 | `content_8ca50876b0df` | REVIEW | High search volume (40,500) and recent impressions above 1,000 produce the maximum score. | The page may already be meeting the underlying search intent effectively. |
| 5 | `content_c841193dc692` | REVIEW | High search volume (40,500) combined with recent impressions above 1,000 gives a maximum score. | High visibility may reflect a healthy page rather than a page needing editorial attention. |
| 6 | `content_eb1510f4b5f1` | REVIEW | High search volume (33,100) and recent impressions of 1,615 produce the maximum score. | The rule does not measure actual content quality, so the page could be performing well without needing changes. |
| 7 | `content_d99d0671553d` | REVIEW | High search volume (22,200) and recent impressions above 1,000 produce the maximum score. | Search demand may be high but the page may already have strong rankings or satisfy the query well. |
| 8 | `content_2db251d1a841` | REVIEW | High search volume (14,800) and extremely high recent impressions (86,434) produce the maximum score. | This is a particularly questionable pick because the very high impressions may indicate the page is already performing strongly. |
| 9 | `content_c35d34131e91` | REVIEW | High search volume (14,800) and recent impressions of 4,547 produce the maximum score. | Strong visibility may mean there is limited benefit from immediate editorial review. |
| 10 | `content_496544bf85aa` | REVIEW | High search volume (14,800) and recent impressions above 1,000 produce the maximum score. | The rule does not account for ranking position, CTR, or content quality, so this may not be a true improvement opportunity. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak pick

A clear weak pick is rank 8: `content_2db251d1a841`.

It receives the maximum score because it has high search volume (14,800) and very high recent impressions (86,434). However, extremely high recent impressions may indicate that the page is already performing strongly rather than being an urgent content opportunity.

This shows a limitation of the baseline: it treats high visibility as a positive opportunity signal but does not distinguish between a page with strong potential and a page that is already performing very well.

### Leakage check

The baseline score uses only `search_volume` and `impressions_last_30d`.

I did not use `trend_pct`, `trend_direction`, future-window performance, or any label-derived field as an input to the score.

In [39]:
# Verify that no leakage fields are used in the scoring inputs
score_inputs = [
    "search_volume",
    "impressions_last_30d"
]

leakage_fields = [
    "trend_pct",
    "trend_direction"
]

print("Score inputs:", score_inputs)
print("Leakage fields excluded:", leakage_fields)

for field in leakage_fields:
    print(f"{field} used in score:", field in score_inputs)

Score inputs: ['search_volume', 'impressions_last_30d']
Leakage fields excluded: ['trend_pct', 'trend_direction']
trend_pct used in score: False
trend_direction used in score: False


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.